# Linux System Slowdown — Diagnosis, Troubleshooting & Verification

**Scenario:** A Linux machine (e.g. a training server, a data-processing box, or your own dev workstation) suddenly *feels slow* — commands lag, the shell is sluggish, everything takes longer than usual.

This notebook documents a complete, real, end‑to‑end workflow:

1. **Create a dummy problem** — a controlled runaway process that mimics a common real-world bug (a stuck loop + an unbounded in-memory cache/leak).
2. **Confirm and diagnose** the slowdown using standard Linux tools.
3. **Troubleshoot** — identify the offending process and resolve it.
4. **Verify** the fix actually worked, with before/after evidence.
5. A short section on **where this fits for an AI/ML Engineer**, and **why an AI engineer needs this skill**.

All command outputs below are **real captured output** from actually running this workflow, not hypothetical text.

## Step 0 — Baseline: system is healthy

In [1]:
!uptime
!free -h

 03:12:14 up 0 min,  0 user,  load average: 0.10, 0.03, 0.01
               total        used        free      shared  buff/cache   available
Mem:           3.9Gi       251Mi       3.7Gi       5.2Mi       112Mi       3.7Gi
Swap:             0B          0B          0B


Baseline is healthy: **load average ≈ 0.10** (well under the 1.0-per-core danger zone on this 1‑core box) and **only ~250 MiB RAM used**.

## Step 1 — Create a dummy problem

We simulate two very common real-world causes of "Linux is slow":

- **A runaway CPU loop** — like a bug in a training loop, a stuck retry loop, or an infinite `while` in a data pipeline that never yields.
- **An unbounded memory leak** — like a cache, buffer, or list in a long-running service (or a Jupyter kernel!) that keeps growing and is never freed.

The script below (`hog.py`) forks into two processes: one spins the CPU for ~40s, the other allocates ~750MB in 5MB chunks and holds it.

In [2]:
hog_script = r"""
import time, os

def burn_cpu():
    end = time.time() + 40
    x = 0
    while time.time() < end:
        x += 1

def leak_memory():
    blob = []
    for _ in range(150):
        blob.append(bytearray(5 * 1024 * 1024))  # ~750MB total
        time.sleep(0.05)
    time.sleep(40)

pid = os.fork()
if pid == 0:
    leak_memory()
    os._exit(0)
else:
    burn_cpu()
"""
with open("hog.py", "w") as f:
    f.write(hog_script)

print("hog.py written.")

hog.py written.


In [3]:
import subprocess, time
proc = subprocess.Popen(["python3", "hog.py"])
print("Launched dummy workload, PID =", proc.pid)
time.sleep(6)  # let it ramp up and actually degrade the system

Launched dummy workload, PID = 519


## Step 2 — Confirm the system is actually slow

In [4]:
!uptime

 03:12:20 up 1 min,  0 user,  load average: 0.26, 0.06, 0.02


The 1-minute load average jumped from **0.10 → 0.26** on a single-core machine (i.e. the CPU run-queue is now meaningfully occupied) — a first, cheap signal that *something* is consuming resources.

## Step 3 — Check CPU usage in detail

In [5]:
!top -bn1 | head -13

top - 03:12:21 up 1 min,  0 user,  load average: 0.26, 0.06, 0.02
Tasks:  53 total,   3 running,  50 sleeping,   0 stopped,   0 zombie
%Cpu(s): 69.2 us, 30.8 sy,  0.0 id,  0.0 wa,  0.0 hi,  0.0 si,  0.0 st
MiB Mem :   3997.8 total,   3432.2 free,    629.5 used,    112.2 buff/cache
MiB Swap:      0.0 total,      0.0 free,      0.0 used.   3368.3 avail Mem

  PID USER      PR  NI    VIRT    RES    SHR S  %CPU  %MEM     TIME+ COMMAND
  519 root      20   0   15236   9544   6548 R  61.5   0.2   0:04.93 python3
  521 root      20   0  399536 389412   3012 R  23.1   9.5   0:01.27 python3
    1 root      20   0   22432   5108   3308 S   0.0   0.1   0:00.57 process_a+
    2 root      20   0       0      0      0 S   0.0   0.0   0:00.00 kthreadd
    3 root      20   0       0      0      0 S   0.0   0.0   0:00.00 pool_work+


`top` confirms **near-100% CPU busy** (69% user + 31% sys) and two `python3` processes sitting at the top — this is our dummy workload.

## Step 4 — Check memory usage

In [6]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:           3.9Gi       613Mi       3.4Gi       5.2Mi       112Mi       3.3Gi
Swap:             0B          0B          0B


Used memory jumped from **251 MiB → 613 MiB**. On a memory-constrained box this trend, if left running, heads toward swapping (or an OOM-kill) — another classic slowdown pattern.

## Step 5 — Identify the offending process(es)

In [7]:
!ps aux --sort=-%cpu | head -6
print("---")
!ps aux --sort=-%mem | head -6

USER       PID %CPU %MEM    VSZ   RSS TTY      STAT START   TIME COMMAND
root       519 79.0  0.2  15236  9544 ?        R    03:12   0:04 python3 hog.py
root       521 20.3  9.5 399536 390312 ?       S    03:12   0:01 python3 hog.py
root         1  0.8  0.1  22432  5108 ?        SLl  03:11   0:00 /process_api --firecracker-init --addr 0.0.0.0:2024 --max-ws-buffer-size 32768 --block-local-connections --listen-vsock-port 2024 --log-vsock-port 5002
root       436  0.4  0.8 2054612 35288 ?       Sl   03:11   0:00 /opt/rclone/rclone-filestore multimount --config /tmp/rclone-mount-config.json
root        12  0.1  0.0      0     0 ?        I    03:11   0:00 [kworker/u4:0-events_unbound]
---
USER       PID %CPU %MEM    VSZ   RSS TTY      STAT START   TIME COMMAND
root       521 20.2  9.5 399536 390312 ?       S    03:12   0:01 python3 hog.py
root       436  0.4  0.8 2054612 35288 ?       Sl   03:11   0:00 /opt/rclone/rclone-filestore multimount --config /tmp/rclone-mount-config.json
root      

**Root cause found:** PID `519` (`python3 hog.py`, the CPU-burn loop) and its child PID `521` (the memory-leak loop) are clearly the top consumers of CPU and RAM respectively.

Other tools that help in real diagnosis (not needed here since the cause is already obvious, but worth knowing):
- `htop` — interactive, color-coded version of the above
- `iostat -x 1` / `vmstat 1` — spot disk I/O bottlenecks (`wa` = time waiting on I/O) instead of CPU/memory
- `dmesg -T | tail` / `journalctl -k -n 50` — check for OOM-killer events or hardware errors
- `iotop` — see which process is hammering the disk
- `df -h` — rule out a full disk (which can also make a system feel "slow")

## Step 6 — Troubleshoot / resolve

Once the offending PID is confirmed, resolve it. Options, from gentlest to most forceful:

1. **`renice`** — if the process is legitimate but low-priority, lower its priority instead of killing it: `renice +10 -p <PID>`
2. **`kill -15 <PID>`** (SIGTERM) — ask the process to shut down cleanly.
3. **`kill -9 <PID>`** (SIGKILL) — force-kill if it won't respond (used below, since this is a deliberately unresponsive dummy loop).
4. If it's a whole misbehaving service, restart it properly: `systemctl restart <service>`.

Here we terminate the dummy workload and its child:

In [8]:
import os, signal, time
os.kill(521, signal.SIGKILL)   # kill child (memory leak)
os.kill(519, signal.SIGKILL)   # kill parent (cpu burn)
time.sleep(2)
print("Sent SIGKILL to PID 519 and 521.")

Sent SIGKILL to PID 519 and 521.


## Step 7 — Verify the problem is resolved

In [9]:
!uptime
!free -h

 03:12:23 up 1 min,  0 user,  load average: 0.24, 0.06, 0.02
               total        used        free      shared  buff/cache   available
Mem:           3.9Gi       340Mi       3.6Gi       5.2Mi       112Mi       3.6Gi
Swap:             0B          0B          0B


In [10]:
!ps aux | grep hog.py | grep -v grep || print("No hog.py process running - confirmed killed.")

No hog.py process running - confirmed killed.


In [11]:
before = {"load_1m": 0.26, "mem_used_mib": 613}
after  = {"load_1m": 0.24, "mem_used_mib": 340}

print(f"Load average 1m : {before['load_1m']} -> {after['load_1m']}")
print(f"Memory used (MiB): {before['mem_used_mib']} -> {after['mem_used_mib']}")

resolved = after["mem_used_mib"] < before["mem_used_mib"] and after["load_1m"] <= before["load_1m"]
print("\nPROBLEM RESOLVED:", resolved)

Load average 1m : 0.26 -> 0.24
Memory used (MiB): 613 -> 340

PROBLEM RESOLVED: True


### Result

- **Memory used** dropped from **613 MiB → 340 MiB** — the leaked ~270–390MB was reclaimed.
- **CPU is idle again** (`ps` shows no `hog.py` process at all).
- **Load average** is already trending back down toward baseline (it decays over a rolling window, so it takes a little longer than CPU/mem to fully settle — this is expected and not a sign of a lingering problem, confirmed by the empty `ps` output).

✅ **The slowdown is resolved and verified with before/after data**, not just "it feels faster.

## Where does this fit for an AI/ML Engineer?

This isn't a "DevOps-only" skill — it shows up directly inside the AI/ML engineering workflow, at layers like:

| Layer | Where "Linux is slow" bites an AI/ML engineer |
|---|---|
| **Local dev / notebooks** | A Jupyter kernel with a runaway cell, an accidental infinite loop, or a variable holding a huge tensor/dataframe that's never released — exactly like the dummy problem above. |
| **Data pipelines / ETL** | A `pandas`/`Dask`/`Spark` job that loads too much into memory, a slow disk (`iostat`/`iowait`) starving the data loader and stalling GPU utilization. |
| **Model training (single node)** | A `DataLoader` with too many workers spawning CPU contention, a memory leak across epochs (e.g., appending metrics/tensors without `.detach()`), CPU preprocessing bottlenecking an idle GPU. |
| **Distributed training / clusters** | One node's OOM-killed process silently stalling an entire distributed job; needing to `ssh` in and diagnose *which* node/process is the straggler. |
| **Model serving / inference in production** | A memory leak in a long-running inference server degrading latency over hours/days; CPU saturation from unbatched requests; needing `top`/`ps`/logs to triage a "the API got slow" incident on-call. |
| **Cloud / containers (Docker, Kubernetes, SageMaker, Vertex AI)** | Container OOM-kills, noisy-neighbor CPU throttling, disk-full errors on a training instance — the same commands (`free`, `ps`, `df`, `dmesg`) are still the first response, just inside a container or pod. |

## Why does an AI Engineer need to resolve this?

1. **Training and inference run *on* Linux.** Nearly every GPU box, cloud VM, Docker container, or Kubernetes pod used for ML is Linux under the hood. Model code doesn't run in a vacuum — it competes for CPU, RAM, disk I/O and (for GPUs) PCIe/host memory with everything else on the box.
2. **Resource bottlenecks silently corrupt "ML" results.** A slow, swapping machine can make a training run look like it has a *code* or *model* problem (slow epochs, dying kernels, `OOMKilled` pods) when the real cause is a leak or a runaway process — exactly the kind of thing this notebook diagnosed.
3. **Time and cost.** GPU instances are expensive per hour. A leaking data loader or a zombie process quietly stealing CPU/RAM extends training wall-clock time and cloud spend for no ML benefit.
4. **Production reliability.** In an on-call rotation for a model-serving endpoint, "the API is slow" is often an OS-level resource issue, not a model issue — an AI engineer who can't diagnose it will misattribute the incident and fix the wrong thing.
5. **Debugging distributed jobs requires it.** In multi-node training, one node's local Linux issue (memory leak, disk full, zombie process) can stall or crash the *entire* distributed job; only host-level troubleshooting (`ps`, `free`, `dmesg`, `journalctl`) pinpoints which node and process is at fault.

In short: **model quality and infrastructure health are entangled.** An AI/ML engineer who can only write model code, but can't triage "why is this Linux box slow," will eventually misdiagnose an infra problem as an ML problem (or vice versa) — costing debugging time, GPU-hours, and production reliability.